# GCP Pipeline Automation - Cloud Functions, Pub/Sub, Dataproc, Composer

**From manual commands to automated, event-driven production pipeline.**

> Prerequisite: Complete [GCP_Full_Pipeline](GCP_Full_Pipeline.ipynb) first. This notebook automates what you built there.

## What We're Building

```mermaid
graph LR
    subgraph "GCP_Full_Pipeline (what you did)"
        M["You typed every\ncommand by hand"]
    end

    subgraph "This Notebook (what we automate)"
        A["File lands in GCS"] -->|"automatic"| B["Cloud Function\nloads to BigQuery"]
        B -->|"automatic"| C["Dataproc Spark\ntransforms Silver"]
        C -->|"automatic"| D["BigQuery SQL\nbuilds Gold"]
        D -->|"automatic"| E["Quality gates\nvalidate"]
        E -->|"pass"| F["Dashboard refreshed"]
        E -->|"fail"| G["Alert sent"]
    end
```

## Architecture: How All Components Connect

```mermaid
graph TD
    subgraph "Event Layer"
        FILE["New file lands\nin GCS bucket"]
        PUBSUB["Pub/Sub\nnotification"]
    end

    subgraph "Trigger Layer"
        CF["Cloud Function\n(serverless)\n- Detects file type\n- Loads into BigQuery"]
    end

    subgraph "Orchestration Layer"
        COMPOSER["Cloud Composer\n(Airflow DAG)\n- Schedules daily at 2 AM\n- Manages dependencies\n- Retries on failure"]
    end

    subgraph "Compute Layer"
        DP_CREATE["Dataproc:\nCreate cluster"]
        DP_SPARK["Dataproc:\nRun PySpark\nSilver transform"]
        DP_DELETE["Dataproc:\nDelete cluster"]
    end

    subgraph "Warehouse Layer"
        BQ_RAW["BigQuery: raw.*\n(Bronze)"]
        BQ_SILVER["BigQuery: silver.*\n(Clean)"]
        BQ_GOLD["BigQuery: gold.*\n(Star schema)"]
    end

    subgraph "Consumer Layer"
        DASH["Looker Studio\n(dashboards)"]
        ML["Vertex AI\n(ML models)"]
        ALERT["Alert\n(oncall)"]
    end

    FILE -->|"triggers"| PUBSUB
    PUBSUB -->|"triggers"| CF
    CF -->|"bq load --append"| BQ_RAW

    COMPOSER -->|"2 AM daily"| DP_CREATE
    DP_CREATE --> DP_SPARK
    DP_SPARK -->|"writes Parquet to GCS"| BQ_SILVER
    DP_SPARK --> DP_DELETE
    BQ_SILVER -->|"SQL transforms"| BQ_GOLD

    BQ_GOLD -->|"quality pass"| DASH
    BQ_GOLD -->|"quality pass"| ML
    BQ_GOLD -->|"quality fail"| ALERT

    style BQ_RAW fill:#cd7f32
    style BQ_SILVER fill:#c0c0c0
    style BQ_GOLD fill:#ffd700
```

## Each Part of This Notebook

```mermaid
graph LR
    subgraph "Part 1: Pub/Sub"
        P1["Create topic\nPublish message\nReceive message"]
    end

    subgraph "Part 2: Cloud Functions"
        P2["Write function\nDeploy\nUpload file\nWatch auto-load"]
    end

    subgraph "Part 3: Dataproc"
        P3["Write PySpark\nCreate cluster\nSubmit job\nVerify output\nDelete cluster"]
    end

    subgraph "Part 4: Composer"
        P4["DAG code\nDependencies\nRetry logic\n(not executable)"]
    end

    P1 --> P2 --> P3 --> P4
```

## What's Executable vs Explained

| Component | Can Run in Colab? | Cost | This Notebook |
|---|---|---|---|
| **Pub/Sub** | Yes | Free tier | Hands-on |
| **Cloud Functions** | Yes (deploy + trigger) | Free tier | Hands-on |
| **Dataproc (Spark)** | Yes (create, submit, delete) | ~$0.50 for 30 min | Hands-on |
| **Cloud Composer** | No ($300/month) | Too expensive for learning | Code shown, explained |


In [ ]:
# === CONFIGURATION ===
# Same project as GCP_Full_Pipeline
PROJECT_ID = "data-pipeline-project-490820"  # <-- Change to your project
BUCKET_NAME = f"de-accelerator-{PROJECT_ID}"
LOCATION = "us-central1"
REGION = "us-central1"

# Datasets (should already exist from GCP_Full_Pipeline)
RAW_DATASET = "raw"
SILVER_DATASET = "silver"
GOLD_DATASET = "gold"

print(f"Project:  {PROJECT_ID}")
print(f"Bucket:   gs://{BUCKET_NAME}/")
print(f"Region:   {REGION}")

In [ ]:
# === AUTHENTICATE ===
import os
if 'COLAB_RELEASE_TAG' in os.environ:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated via Colab.")
else:
    print("Running locally. Use 'gcloud auth login' if not authenticated.")

!gcloud config set project {PROJECT_ID}

# Enable required APIs
!gcloud services enable \
    cloudfunctions.googleapis.com \
    pubsub.googleapis.com \
    dataproc.googleapis.com \
    cloudbuild.googleapis.com \
    run.googleapis.com \
    2>&1 | tail -3

print("APIs enabled.")

---

## Part 1: Pub/Sub - The Notification Layer

### What is Pub/Sub?

A message queue. One service PUBLISHES a message. Another service SUBSCRIBES and receives it.

**Analogy:** A bulletin board. Someone pins a note (publish). Anyone watching the board (subscriber) sees it.

**In the pipeline:** GCS publishes "a new file arrived." Your Cloud Function subscribes and gets triggered.

```
GCS bucket          Pub/Sub topic        Cloud Function
(file arrives) ---> (message posted) ---> (code runs)
```

### Why not trigger the function directly from GCS?

You can (and we will). But Pub/Sub adds flexibility:
- Multiple subscribers can react to the same event
- Messages are retained if the subscriber is down (retry built in)
- You can filter messages (only react to certain file types)

### AWS equivalent: SNS (notification) + SQS (queue)

In [ ]:
# === PUB/SUB: Create a topic and subscription ===

TOPIC_NAME = "pipeline-file-events"
SUBSCRIPTION_NAME = "pipeline-file-events-sub"

# Create topic
!gcloud pubsub topics create {TOPIC_NAME} 2>&1 || echo "Topic exists."

# Create subscription
!gcloud pubsub subscriptions create {SUBSCRIPTION_NAME} \
    --topic={TOPIC_NAME} 2>&1 || echo "Subscription exists."

print(f"Topic: {TOPIC_NAME}")
print(f"Subscription: {SUBSCRIPTION_NAME}")

In [ ]:
# === PUB/SUB: Publish a test message ===
# Simulates what GCS would send when a file arrives

import json

test_message = json.dumps({
    "bucket": BUCKET_NAME,
    "name": "bronze/calls/calls_2026_03_15.json",
    "size": "125000",
    "timeCreated": "2026-03-15T02:00:00Z"
})

!gcloud pubsub topics publish {TOPIC_NAME} --message='{test_message}'

print(f"Published message to {TOPIC_NAME}")

In [ ]:
# === PUB/SUB: Receive the message ===
# Pull the message we just published

!gcloud pubsub subscriptions pull {SUBSCRIPTION_NAME} --auto-ack --limit=5

# You Should See: The JSON message we published above
# This is what your Cloud Function would receive when a file lands in GCS

You just saw the full Pub/Sub cycle: create topic, publish message, receive message. In production, GCS publishes automatically and your Cloud Function receives automatically. No manual steps.

---

## Part 2: Cloud Functions - The Trigger Layer

### What is a Cloud Function?

A small piece of code that runs when an event happens. No servers to manage. You deploy it and Google runs it when triggered.

**Analogy:** A doorbell. Press it (event) and the bell rings (code runs). You don't keep someone standing at the door.

**In the pipeline:** When a file lands in GCS, the function loads it into BigQuery.

### AWS equivalent: Lambda

In [ ]:
# === CLOUD FUNCTION: Write the function code ===
# We write the function to a local file, then deploy it

import os
os.makedirs("/tmp/pipeline-function", exist_ok=True)

# main.py - the function code
function_code = '''
from google.cloud import bigquery
import json

def process_new_file(event, context):
    """
    Triggered by a new file in GCS.
    Loads the file into the appropriate BigQuery table.
    """
    bucket = event["bucket"]
    file_name = event["name"]
    
    # Only process files in the bronze/ folder
    if not file_name.startswith("bronze/"):
        print(f"Skipping {file_name} (not in bronze/)")
        return
    
    print(f"Processing: gs://{bucket}/{file_name}")
    
    # Map file path to BigQuery table and format
    if "calls" in file_name:
        table_id = "raw.calls"
        source_format = bigquery.SourceFormat.NEWLINE_DELIMITED_JSON
    elif "orders" in file_name:
        table_id = "raw.orders"
        source_format = bigquery.SourceFormat.CSV
    elif "campaigns" in file_name:
        table_id = "raw.campaigns"
        source_format = bigquery.SourceFormat.CSV
    elif "products" in file_name:
        table_id = "raw.products"
        source_format = bigquery.SourceFormat.CSV
    else:
        print(f"Unknown file type: {file_name}")
        return
    
    # Load into BigQuery (APPEND, not replace)
    client = bigquery.Client()
    job_config = bigquery.LoadJobConfig(
        source_format=source_format,
        autodetect=True,
        write_disposition="WRITE_APPEND",
    )
    
    uri = f"gs://{bucket}/{file_name}"
    load_job = client.load_table_from_uri(uri, table_id, job_config=job_config)
    load_job.result()  # Wait for completion
    
    print(f"Loaded {load_job.output_rows} rows into {table_id}")
    return f"Success: {file_name} -> {table_id}"
'''

# requirements.txt
requirements = 'google-cloud-bigquery>=3.0.0\n'

with open("/tmp/pipeline-function/main.py", "w") as f:
    f.write(function_code)
with open("/tmp/pipeline-function/requirements.txt", "w") as f:
    f.write(requirements)

print("Function code written to /tmp/pipeline-function/")
print("\n--- main.py ---")
print(function_code)

In [ ]:
# === CLOUD FUNCTION: Deploy ===
# This deploys the function to GCP. It triggers on any file created in your bucket.
#
# IMPORTANT: This uses Cloud Functions Gen 2 (recommended).
# First deploy takes 2-3 minutes.

!gcloud functions deploy process_new_file \
    --gen2 \
    --runtime=python311 \
    --region={REGION} \
    --source=/tmp/pipeline-function \
    --entry-point=process_new_file \
    --trigger-event-filters="type=google.cloud.storage.object.v1.finalized" \
    --trigger-event-filters="bucket={BUCKET_NAME}" \
    --memory=256MB \
    --timeout=120s \
    2>&1 | tail -10

print("\nFunction deployed. It will trigger when any file lands in your GCS bucket.")

In [ ]:
# === CLOUD FUNCTION: Test it ===
# Upload a file to GCS. The function should trigger and load it into BigQuery.

# First, check current row count in raw.calls
import subprocess
def run_bq(sql):
    result = subprocess.run(["bq", "query", "--use_legacy_sql=false", "--format=prettyjson", sql],
                            capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0: print("ERROR:", result.stderr)

print("Row count BEFORE:")
run_bq(f"SELECT COUNT(*) as row_count FROM `{PROJECT_ID}.raw.calls`")

# Upload the same calls.json again (simulating new data arriving)
!gcloud storage cp /tmp/de-repo/data/calls.json gs://{BUCKET_NAME}/bronze/calls/calls_test_trigger.json

print("\nFile uploaded. Cloud Function should trigger in 10-30 seconds...")
print("Wait 30 seconds, then run the next cell to check.")

In [ ]:
# === VERIFY: Did the Cloud Function load the data? ===
# Run this 30 seconds after uploading the test file

print("Row count AFTER (should be higher):")
run_bq(f"SELECT COUNT(*) as row_count FROM `{PROJECT_ID}.raw.calls`")

# Check function logs
print("\n--- Cloud Function Logs (last 5 entries) ---")
!gcloud functions logs read process_new_file --gen2 --region={REGION} --limit=5 2>&1

**What just happened:**
1. You uploaded a file to GCS
2. GCS sent a notification to your Cloud Function
3. The function detected the file type (calls = JSON)
4. The function loaded it into BigQuery `raw.calls` (append)
5. All automatic. No commands typed.

This is how Bronze ingestion works in production. Files arrive, the function loads them. 24/7, no human needed.

---

## Part 3: Dataproc (Spark) - The Transform Layer

### Why Spark for Silver?

The BigQuery SQL approach from GCP_Full_Pipeline works fine for our 500-row dataset. At scale (millions of rows, complex transforms, ML features), Spark distributes the work across many machines.

**Same logic (dedup, clean, type-fix). Different execution engine.**

```
BigQuery SQL (what we did):       Spark/Dataproc (what we do now):
  CREATE TABLE silver.calls AS      spark.read.json("gs://bronze/")
  SELECT ... FROM raw.calls         .withColumn(dedup, clean)
  WHERE row_num = 1                 .write.parquet("gs://silver/")
```

### AWS equivalent: EMR (Elastic MapReduce)

In [ ]:
# === DATAPROC: Write the PySpark Silver transform ===

spark_code = '''
# silver_transform.py
# Run on Dataproc: reads Bronze from GCS, writes Silver to GCS as Parquet

import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, row_number, lower, trim, when, lit, hour,
    from_utc_timestamp, date_format
)
from pyspark.sql.window import Window

# Get bucket from command line argument
BUCKET = sys.argv[1] if len(sys.argv) > 1 else "gs://de-accelerator-default"
print(f"Bucket: {BUCKET}")

spark = SparkSession.builder.appName("SilverTransform").getOrCreate()

# =============================================
# READ BRONZE (raw files from GCS)
# =============================================
calls = spark.read.json(f"{BUCKET}/bronze/calls.json")
orders = spark.read.option("header", True).option("inferSchema", True) \\
    .csv(f"{BUCKET}/bronze/orders.csv")

print(f"Bronze calls: {calls.count()} rows")
print(f"Bronze orders: {orders.count()} rows")

# =============================================
# SILVER: Clean calls
# Same logic as BigQuery SQL, but in PySpark
# =============================================

# Dedup by call_id (keep first by start_time)
window = Window.partitionBy("call_id").orderBy("start_time")

silver_calls = (calls
    .withColumn("row_num", row_number().over(window))
    .filter(col("row_num") == 1)
    .drop("row_num")

    # Timezone: UTC to Eastern
    .withColumn("call_started_local",
                from_utc_timestamp(col("start_time"), "US/Eastern"))
    .withColumn("call_ended_local",
                from_utc_timestamp(col("end_time"), "US/Eastern"))
    .withColumn("call_date_local",
                date_format(col("call_started_local"), "yyyy-MM-dd"))
    .withColumn("call_hour_local",
                hour(col("call_started_local")))

    # Standardize text
    .withColumn("disposition", lower(trim(col("disposition"))))
    .withColumn("call_type", trim(col("channel")))

    # Flag nulls (don't drop)
    .withColumn("has_missing_duration", col("duration").isNull())
    .withColumn("has_missing_disposition", col("disposition").isNull())
)

# SILVER: Clean orders
window_orders = Window.partitionBy("order_id").orderBy(col("order_date").desc())
silver_orders = (orders
    .withColumn("row_num", row_number().over(window_orders))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

# =============================================
# WRITE SILVER to GCS as Parquet
# =============================================
silver_calls.write.mode("overwrite") \\
    .partitionBy("call_date_local") \\
    .parquet(f"{BUCKET}/silver/calls/")

silver_orders.write.mode("overwrite") \\
    .parquet(f"{BUCKET}/silver/orders/")

print(f"Silver calls: {silver_calls.count()} rows")
print(f"Silver orders: {silver_orders.count()} rows")
print("Silver transform complete.")

spark.stop()
'''

with open("/tmp/silver_transform.py", "w") as f:
    f.write(spark_code)

# Upload to GCS
!gcloud storage cp /tmp/silver_transform.py gs://{BUCKET_NAME}/code/

print("PySpark Silver transform uploaded to GCS.")
print("\n--- silver_transform.py ---")
print(spark_code[:500] + "\n...")

In [ ]:
# === DATAPROC: Create a cluster ===
# This takes 2-3 minutes. Cost: ~$0.15 for 30 minutes with this config.

CLUSTER_NAME = "pipeline-silver-cluster"

!gcloud dataproc clusters create {CLUSTER_NAME} \
    --region={REGION} \
    --num-workers=2 \
    --worker-machine-type=n1-standard-2 \
    --master-machine-type=n1-standard-2 \
    --image-version=2.1-debian11 \
    --max-idle=600s \
    2>&1 | tail -5

# --max-idle=600s: auto-delete after 10 minutes of no jobs (cost protection)

print(f"\nCluster '{CLUSTER_NAME}' created.")

In [ ]:
# === DATAPROC: Submit the Silver transform job ===

!gcloud dataproc jobs submit pyspark \
    gs://{BUCKET_NAME}/code/silver_transform.py \
    --cluster={CLUSTER_NAME} \
    --region={REGION} \
    -- gs://{BUCKET_NAME} \
    2>&1 | tail -20

print("\nSpark job complete.")

In [ ]:
# === VERIFY: Check Silver output in GCS ===

print("Silver calls (Parquet files in GCS):")
!gcloud storage ls gs://{BUCKET_NAME}/silver/calls/ 2>&1 | head -10

print("\nSilver orders (Parquet files in GCS):")
!gcloud storage ls gs://{BUCKET_NAME}/silver/orders/ 2>&1 | head -5

# Load Silver Parquet into BigQuery for Gold layer
print("\nLoading Silver Parquet into BigQuery...")
!bq load --source_format=PARQUET --replace \
    {SILVER_DATASET}.calls \
    'gs://{BUCKET_NAME}/silver/calls/*.parquet' 2>&1 | tail -3

!bq load --source_format=PARQUET --replace \
    {SILVER_DATASET}.orders \
    'gs://{BUCKET_NAME}/silver/orders/*.parquet' 2>&1 | tail -3

print("\nSilver data loaded into BigQuery. Ready for Gold.")

In [ ]:
# === DATAPROC: Delete the cluster (stop paying) ===
# IMPORTANT: Always delete when done.

!gcloud dataproc clusters delete {CLUSTER_NAME} \
    --region={REGION} \
    --quiet \
    2>&1 | tail -3

print(f"Cluster '{CLUSTER_NAME}' deleted. No more charges.")

**What just happened:**
1. You wrote a PySpark script (same dedup/clean logic as BigQuery SQL)
2. Uploaded it to GCS
3. Created a Dataproc cluster (2 workers)
4. Submitted the job (Spark distributed the work)
5. Silver Parquet files written to GCS
6. Loaded Silver Parquet into BigQuery (for Gold queries)
7. Deleted the cluster (stop paying)

**The difference from GCP_Full_Pipeline:** There, Silver was created by BigQuery SQL (`CREATE TABLE silver.calls AS SELECT ...`). Here, Silver was created by Spark writing Parquet to GCS, then loaded into BigQuery. Same result. Different engine. The Spark path scales to terabytes.

---

## Part 4: Cloud Composer (Airflow) - The Orchestration Layer

### Why We Can't Run This in Colab

Cloud Composer is a managed Airflow environment. The smallest configuration costs ~$300/month. That's not practical for learning.

**What we'll do:** Show the DAG code that would orchestrate our pipeline, explain each piece, and demonstrate how it ties everything together.

### For hands-on Airflow practice:
- Run Airflow locally with Docker: `docker compose up` (free)
- Use Astronomer's free dev tool: `astro dev start`
- Deploy to Cloud Composer only for production

### AWS equivalent: MWAA (Managed Airflow)

In [ ]:
# === CLOUD COMPOSER: The DAG (Airflow workflow definition) ===
# This code runs IN Airflow, not in this notebook.
# Showing it here so you can see the full orchestration.

dag_code = '''
# dags/call_center_pipeline.py
# Deploy this to Cloud Composer (or local Airflow)

from airflow import DAG
from airflow.providers.google.cloud.operators.bigquery import BigQueryInsertJobOperator
from airflow.providers.google.cloud.operators.dataproc import (
    DataprocCreateClusterOperator,
    DataprocSubmitJobOperator,
    DataprocDeleteClusterOperator,
)
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta

PROJECT_ID = "your-project-id"
REGION = "us-central1"
BUCKET = "your-bucket"
CLUSTER_NAME = "pipeline-silver-cluster"

default_args = {
    "retries": 2,
    "retry_delay": timedelta(minutes=5),
    "email_on_failure": True,
    "email": ["oncall@company.com"],
}

dag = DAG(
    "call_center_pipeline",
    description="Full pipeline: Bronze -> Silver (Spark) -> Gold (BigQuery)",
    schedule_interval="0 2 * * *",   # 2 AM daily
    start_date=datetime(2026, 3, 1),
    catchup=False,
    default_args=default_args,
)

# ---- Task 1: Create Dataproc cluster ----
create_cluster = DataprocCreateClusterOperator(
    task_id="create_cluster",
    project_id=PROJECT_ID,
    region=REGION,
    cluster_name=CLUSTER_NAME,
    cluster_config={
        "master_config": {"num_instances": 1, "machine_type_uri": "n1-standard-2"},
        "worker_config": {"num_instances": 2, "machine_type_uri": "n1-standard-2"},
    },
    dag=dag,
)

# ---- Task 2: Run Silver transform on Spark ----
silver_transform = DataprocSubmitJobOperator(
    task_id="silver_transform",
    project_id=PROJECT_ID,
    region=REGION,
    job={
        "placement": {"cluster_name": CLUSTER_NAME},
        "pyspark_job": {
            "main_python_file_uri": f"gs://{BUCKET}/code/silver_transform.py",
            "args": [f"gs://{BUCKET}"],
        },
    },
    dag=dag,
)

# ---- Task 3: Delete cluster (cost control) ----
delete_cluster = DataprocDeleteClusterOperator(
    task_id="delete_cluster",
    project_id=PROJECT_ID,
    region=REGION,
    cluster_name=CLUSTER_NAME,
    trigger_rule="all_done",  # Delete even if transform failed
    dag=dag,
)

# ---- Task 4: Load Silver Parquet into BigQuery ----
load_silver_to_bq = BigQueryInsertJobOperator(
    task_id="load_silver_to_bq",
    configuration={
        "load": {
            "sourceUris": [f"gs://{BUCKET}/silver/calls/*.parquet"],
            "destinationTable": {
                "projectId": PROJECT_ID,
                "datasetId": "silver",
                "tableId": "calls",
            },
            "sourceFormat": "PARQUET",
            "writeDisposition": "WRITE_TRUNCATE",
        }
    },
    dag=dag,
)

# ---- Task 5: Build Gold marts ----
build_gold = BigQueryInsertJobOperator(
    task_id="build_gold_marts",
    configuration={
        "query": {
            "query": open("/home/airflow/dags/sql/gold_marts.sql").read(),
            "useLegacySql": False,
        }
    },
    dag=dag,
)

# ---- Task 6: Quality gates ----
def run_quality_gates():
    from google.cloud import bigquery
    client = bigquery.Client()
    
    # Gate: no duplicates in fact table
    dupes = list(client.query("""
        SELECT call_id, COUNT(*) as cnt
        FROM gold.fact_calls GROUP BY call_id HAVING cnt > 1
    """).result())
    
    if dupes:
        raise ValueError(f"QUALITY GATE FAILED: {len(dupes)} duplicate call_ids")
    print("All gates passed.")

quality_gates = PythonOperator(
    task_id="quality_gates",
    python_callable=run_quality_gates,
    dag=dag,
)

# ---- Dependencies (the DAG) ----
# This defines the execution order:
#
#   create_cluster -> silver_transform -> delete_cluster
#                                     -> load_silver_to_bq -> build_gold -> quality_gates

create_cluster >> silver_transform >> [delete_cluster, load_silver_to_bq]
load_silver_to_bq >> build_gold >> quality_gates
'''

print("=" * 60)
print("AIRFLOW DAG: call_center_pipeline")
print("=" * 60)
print("\nThis DAG runs at 2 AM daily and orchestrates:")
print("  1. Create Dataproc cluster")
print("  2. Run Silver transform (Spark)")
print("  3. Delete cluster (cost control)")
print("  4. Load Silver Parquet into BigQuery")
print("  5. Build Gold marts (BigQuery SQL)")
print("  6. Quality gates (fail = alert oncall)")
print("\n--- Execution flow ---")
print("create_cluster >> silver_transform >> delete_cluster")
print("                                  >> load_silver_to_bq >> build_gold >> quality_gates")
print("\nIf silver_transform fails:")
print("  - Retries 2 times (5 min apart)")
print("  - If still fails: cluster gets deleted (trigger_rule=all_done), email sent")
print("  - Gold does NOT run (stale data is worse than no data)")
print("\n--- Full DAG code shown above. Deploy to Cloud Composer or local Airflow. ---")

---

## Part 5: The Complete Automated Pipeline

### How Everything Connects in Production

```mermaid
graph TD
    subgraph "Daytime: Event-Driven Bronze"
        A["Source system\ndrops file in GCS"] -->|"GCS notification"| B["Pub/Sub"]
        B -->|"triggers"| C["Cloud Function"]
        C -->|"bq load --append"| D["BigQuery: raw.*\n(Bronze)"]
    end

    subgraph "2 AM: Scheduled Transform"
        E["Cloud Composer\nDAG triggers"] --> F["Create Dataproc\ncluster"]
        F --> G["Submit PySpark\nSilver transform"]
        G --> H["Delete cluster\n(cost control)"]
        G -->|"Parquet to GCS"| I["Load Silver into\nBigQuery"]
        I --> J["Build Gold marts\n(BigQuery SQL)"]
    end

    subgraph "Validation"
        J --> K{"Quality\ngates"}
        K -->|"Pass"| L["Dashboard\nrefreshed"]
        K -->|"Pass"| M["ML model\nretrained"]
        K -->|"Fail"| N["Alert oncall\nengineer"]
    end

    D -.->|"data ready for\nnext scheduled run"| E

    style D fill:#cd7f32
    style I fill:#c0c0c0
    style J fill:#ffd700
```

### The Two Rhythms of a Production Pipeline

| Rhythm | When | What | Component |
|---|---|---|---|
| **Event-driven** | Anytime a file arrives | Bronze load (append new data) | Cloud Function |
| **Scheduled** | 2 AM daily | Silver transform, Gold build, quality gates | Cloud Composer + Dataproc |

Bronze loads happen throughout the day as files arrive. The heavy transforms run once on a schedule.

### What You Built in This Notebook

| Component | What You Did | Production Role |
|---|---|---|
| **Pub/Sub** | Created topic, published, received message | Notification layer between services |
| **Cloud Functions** | Deployed function, triggered by file upload | Automated Bronze ingestion |
| **Dataproc** | Created cluster, submitted Spark job, deleted cluster | Scalable Silver transforms |
| **Cloud Composer** | Reviewed the DAG code | Orchestrates the entire flow |

### GCP to AWS Translation

| GCP | AWS | Role |
|---|---|---|
| Cloud Functions | Lambda | Event-driven triggers |
| Pub/Sub | SNS + SQS | Messaging |
| Dataproc | EMR | Managed Spark |
| Cloud Composer | MWAA | Managed Airflow |
| GCS | S3 | File storage |
| BigQuery | Redshift | Data warehouse |


In [ ]:
# === CLEANUP ===
# Uncomment to delete everything created in this notebook

# Delete Cloud Function
# !gcloud functions delete process_new_file --gen2 --region={REGION} --quiet

# Delete Pub/Sub
# !gcloud pubsub subscriptions delete {SUBSCRIPTION_NAME} --quiet
# !gcloud pubsub topics delete {TOPIC_NAME} --quiet

# Delete Dataproc cluster (if still running)
# !gcloud dataproc clusters delete {CLUSTER_NAME} --region={REGION} --quiet

# Delete test file
# !gcloud storage rm gs://{BUCKET_NAME}/bronze/calls/calls_test_trigger.json

print("Uncomment the lines above to clean up resources.")